In [2]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


In [2]:
from google.colab import drive
drive.mount('/content/drive')
!ls

Mounted at /content/drive
drive  sample_data


In [3]:
%cd ./drive/MyDrive/ColabNotebooks/Definitive

/content/drive/MyDrive/ColabNotebooks/Definitive


In [4]:
!ls

512.pgm		 dftnaive  output_cuda-512.pgm
DFTcudanaive.cu  dftopt1   report.ncu-rep
DFTcudaOpt1.cu	 out.pgm   test_report_opt1.ncu-rep


# Compilazione soluzione naive


In [11]:
!nvcc -O3 -lineinfo DFTcudanaive.cu -o dftnaive

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudanaive.cu(51): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudanaive.cu(57): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudanaive.cu(58): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudanaive.cu(62): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudanaive.cu(51): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudanaive.cu(57): warning #1650-D: result of call is not used


In [12]:
!./dftnaive 512.pgm

Immagine caricata di dimensioni H:288 W:512
Trasformata e antitrasformata completate. Risultato salvato in output_cuda_naive-512.pgm


# Esecuzione della soluzione naive 
### Baseline del progetto, realizzata concentrandosi sul solo funzionamento della soluzione e non sull'efficienza
"-f" fa sovrascrittura del file nsight se gia presente su drive

--set basic / --set full

In [9]:
!ncu --set basic -f -o test_report_naive ./dftnaive 512.pgm

Immagine caricata di dimensioni H:288 W:512
==PROF== Connected to process 2434 (/content/drive/MyDrive/ColabNotebooks/Definitive/dftnaive)
==PROF== Profiling "dft" - 0: 0%..
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
..50%....100% - 9 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 9 passes
==PROF== Profiling "idft" - 2: 0%....50%....100% - 9 passes
Trasformata e antitrasformata completate. Risultato salvato in output_cuda-512.pgm
==PROF== Disconnected from process 2434
==PROF== Report: /content/drive/MyDrive/ColabNotebooks/Definitive/test_report_naive.ncu-rep


# Prima ottimizzazione: 
### sono stati usati stratagemmi per ottimizzare l'efficienza dei calcoli matematici
- sincosf
- riduzione della precisione da double a float
- precalcolo delle componenti sin/cos per verticale (costante su u)
- uso di fmaf_rn per operazioni fma non riconosciute come tali dal compilatore e  "-use_fast_math" come argomento del compilatore per riconoscere operazioni adatte al fused multiply add(FMA) e sostituire "sincosf" con un'operazione di basso livello hardware "__sincosf", piu veloce a discapito di una leggera perdita di precisione 


In [13]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt1.cu -o dftopt1

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt1.cu(51): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt1.cu(57): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt1.cu(58): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt1.cu(62): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt1.cu(51): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt1.cu(57): warning #1650-D: result of call is not used
      

esecuzione test

In [14]:
!./dftopt1 512.pgm 

Immagine caricata di dimensioni H:288 W:512
Trasformata e antitrasformata completate. Risultato salvato in output_cuda_opt1-512.pgm


### Esecuzione con salvataggio dati per nsight 

In [ ]:
!ncu --set full -f -o test_report_opt1 ./dftopt1 512.pgm 

Immagine caricata di dimensioni H:288 W:512
==PROF== Connected to process 17623 (/content/drive/MyDrive/ColabNotebooks/Definitive/dftopt1)
==PROF== Profiling "dft_opt1" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft_opt1" - 2: 0%....50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato in output_cuda_opt1-512.pgm
==PROF== Disconnected from process 17623
==PROF== Report: /content/drive/MyDrive/ColabNotebooks/Definitive/test_report_opt1.ncu-rep


# Seconda ottimizzazione:
### Uso della shared memory
Viene fornitaa shared memory a griglie 16x16 di thread

In [26]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt2.cu -o dftopt2

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt2.cu:84: warning: "PI" redefined
   84 | #define PI 3.14159265358979323846f
      | 
DFTcudaOpt2.cu:7: note: this is the location of the previous definition
    7 | #define PI 3.14159265358979323846
      | 
DFTcudaOpt2.cu(52): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt2.cu(58): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt2.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt2.cu(63): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt2.cu:84: warning: "PI" redef

esecuzione test

In [8]:
!./dftopt2 Bologna-512.pgm

Immagine caricata di dimensioni H:340 W:512
Trasformata e antitrasformata completate. Risultato salvato in output_cuda-512.pgm


### Esecuzione con salvataggio dati per nsight

In [10]:
!ncu --set full -f -o test_report_opt2 ./dftopt2 Bologna-512.pgm

Immagine caricata di dimensioni H:340 W:512
==PROF== Connected to process 15189 (/content/drive/MyDrive/ColabNotebooks/dftopt2)
==PROF== Profiling "dft2D_opt_shared" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft2D_opt_shared" - 2: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato 

# Terza ottimizzazione
## Vari test con loop unrolling
l'unrolling viene fatto specificando al compilatore tramite keyword #pragma unroll 'n', dove n è il numero di interazioni in cui spezzare il loop, nel nostro caso operando sul loop interno di dft e idft in seguito alla definizione dei tile come 16x16, faremo unroll per divisore di 16, ovvero 4, 8 e 16


# Test 1: unroll 4



In [27]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt3.cu -o dftopt3

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt3.cu:84: warning: "PI" redefined
   84 | #define PI 3.14159265358979323846f
      | 
DFTcudaOpt3.cu:7: note: this is the location of the previous definition
    7 | #define PI 3.14159265358979323846
      | 
DFTcudaOpt3.cu(52): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt3.cu(58): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt3.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt3.cu(63): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt3.cu:84: warning: "PI" redef

test:

In [18]:
!./dftopt3 Bologna-512.pgm

Immagine caricata di dimensioni H:340 W:512
Trasformata e antitrasformata completate. Risultato salvato in output_cuda-512.pgm


esecuzione con raccolta report

In [19]:
!ncu --set full -f -o test_report_opt3 ./dftopt3 Bologna-512.pgm

Immagine caricata di dimensioni H:340 W:512
==PROF== Connected to process 18218 (/content/drive/MyDrive/ColabNotebooks/dftopt3)
==PROF== Profiling "dft2D_opt_shared" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft2D_opt_shared" - 2: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato 

# Test 2: unroll 8



In [28]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt4.cu -o dftopt4

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt4.cu:84: warning: "PI" redefined
   84 | #define PI 3.14159265358979323846f
      | 
DFTcudaOpt4.cu:7: note: this is the location of the previous definition
    7 | #define PI 3.14159265358979323846
      | 
DFTcudaOpt4.cu(52): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt4.cu(58): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt4.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt4.cu(63): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt4.cu:84: warning: "PI" redef

test:

In [ ]:
!./dftopt4 Bologna-512.pgm

esecuzione con raccolta report

In [21]:
!ncu --set full -f -o test_report_opt4 ./dftopt4 Bologna-512.pgm

Immagine caricata di dimensioni H:340 W:512
==PROF== Connected to process 18713 (/content/drive/MyDrive/ColabNotebooks/dftopt4)
==PROF== Profiling "dft2D_opt_shared" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft2D_opt_shared" - 2: 0%....50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato in output_cuda-512.pgm
==PROF== Disconnected from process 18713
==PROF== Report: /content/drive/MyDrive/ColabNotebooks/test_report_opt4.ncu-rep


# Test 3: unroll 16



In [29]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt5.cu -o dftopt5

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt5.cu:84: warning: "PI" redefined
   84 | #define PI 3.14159265358979323846f
      | 
DFTcudaOpt5.cu:7: note: this is the location of the previous definition
    7 | #define PI 3.14159265358979323846
      | 
DFTcudaOpt5.cu(52): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt5.cu(58): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt5.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt5.cu(63): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt5.cu:84: warning: "PI" redef

test:

In [ ]:
!./dftopt5 Bologna-512.pgm

esecuzione con raccolta report

In [30]:
!ncu --set full -f -o test_report_opt5 ./dftopt5 Bologna-512.pgm

Immagine caricata di dimensioni H:340 W:512
==PROF== Connected to process 4192 (/content/drive/MyDrive/ColabNotebooks/dftopt5)
==PROF== Profiling "dft2D_opt_shared" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft2D_opt_shared" - 2: 0%....50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato in output_cuda-512.pgm
==PROF== Disconnected from process 4192
==PROF== Report: /content/drive/MyDrive/ColabNotebooks/test_report_opt5.ncu-rep


### Si è notato un incremento delle performance solo nel momento in cui è stato fatto loop unrolling di 16 istruzione, dato che va a arimuovere completamente l'operazione condizionale del ciclo

# Quarta ottimizzazione
### Cache L1 configuration

## _ _restrict__ sui puntatori nella firma dei kernel:
- dice al compilatore che in e out non si sovrappongono in memoria
## Vector store con float2 per la DFT:
- La scrittura di MyComplex (due float) può essere fatta come store vettoriale se la struct è allineata a 8 byte — una singola transazione invece di due, (IMPORTANTE specificare la keyword align(8) davanti alla struct mycomplex per migliorare la coalescenza)

In [72]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt6.cu -o dftopt6

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt6.cu:84: warning: "PI" redefined
   84 | #define PI 3.14159265358979323846f
      | 
DFTcudaOpt6.cu:7: note: this is the location of the previous definition
    7 | #define PI 3.14159265358979323846
      | 
DFTcudaOpt6.cu(52): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt6.cu(58): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt6.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt6.cu(63): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt6.cu:84: warning: "PI" redef

testing

In [ ]:
!./dftopt6 Bologna-512.pgm

### profiling

In [73]:
!ncu --set full -f -o test_report_opt6 ./dftopt6 Bologna-512.pgm

Immagine caricata di dimensioni H:340 W:512
==PROF== Connected to process 14937 (/content/drive/MyDrive/ColabNotebooks/dftopt6)
==PROF== Profiling "dft2D_opt_L1" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft2D_opt_L1" - 2: 0%....50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato in output_cuda-512.pgm
==PROF== Disconnected from process 14937
==PROF== Report: /content/drive/MyDrive/ColabNotebooks/test_report_opt6.ncu-rep
